In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Load both datasets
df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\dropped_new_data.csv")
# Preview the result
print(df.head())

  payment_type  profit_per_order  sales_per_customer         category_name  \
0      PAYMENT        101.010895           195.02570  Indoor/Outdoor Games   
1     TRANSFER         85.423610           245.20793  Indoor/Outdoor Games   
2      PAYMENT        261.173770           456.55527      Cardio Equipment   
3        DEBIT        -52.374670           191.65901          Water Sports   
4        DEBIT         55.085342           187.45561          Water Sports   

  customer_segment department_name   latitude  longitude  market  \
0      Home Office        Fan Shop  41.478510 -87.972565  Europe   
1         Consumer        Fan Shop  18.281605 -66.370510   LATAM   
2         Consumer        Footwear  18.281320 -71.919000  Europe   
3         Consumer        Fan Shop  18.289013 -66.370520   LATAM   
4         Consumer        Fan Shop  18.227660 -66.370530    USCA   

      order_city  ...   shipping_mode  label  dest_latitude  dest_longitude  \
0          Viena  ...  Standard Class      

In [4]:
df.nunique()

payment_type                      4
profit_per_order              15431
sales_per_customer             8939
category_name                    50
customer_segment                  3
department_name                  11
latitude                      14977
longitude                      9931
market                            5
order_city                     2755
order_country                   149
order_date                    15116
order_item_discount            1236
order_item_discount_rate        244
order_item_product_price        392
order_item_profit_ratio        1959
order_item_quantity               6
sales                           472
order_item_total_amount        1560
order_profit_per_order        10341
order_region                     23
order_state                     899
order_status                      7
product_name                    116
shipping_date                 15122
shipping_mode                     4
label                             3
dest_latitude               

DERIVATION OF FEATURES

In [6]:
import pandas as pd

# Define origin date
origin = pd.Timestamp('1990-01-01')

# Convert numeric to datetime
df['order_date'] = origin + pd.to_timedelta(df['order_date'], unit='D')
df['shipping_date'] = origin + pd.to_timedelta(df['shipping_date'], unit='D')

In [7]:
df['order_to_shipment_days'] = (df['shipping_date'] - df['order_date']).dt.days
df['order_dayofweek'] = df['order_date'].dt.dayofweek
df['shipping_dayofweek'] = df['shipping_date'].dt.dayofweek
df['order_hour'] = df['order_date'].dt.hour
df['shipping_hour'] = df['shipping_date'].dt.hour
# Step 2: Difference in minutes
df["order_shipping_time"] = (df["shipping_date"] - df["order_date"]).dt.total_seconds() / 60

# Step 3 (optional): Also in hours and days
df["order_shipping_time"] = df["order_shipping_time"] / 60

In [8]:
def get_daypart(hour):
    if 4 <= hour <= 7:
        return 'Early Morning'
    elif 8 <= hour <= 11:
        return 'Morning'
    elif 12 <= hour <= 15:
        return 'Noon'
    elif 16 <= hour <= 19:
        return 'Eve'
    elif 20 <= hour <= 23:
        return 'Night'
    else:
        return 'Late Night'

df['order_daypart'] = df['order_hour'].apply(get_daypart)
df['ship_daypart'] = df['shipping_hour'].apply(get_daypart)


In [9]:
daypart_map = {
    'Early Morning': 0,
    'Morning': 1,
    'Noon': 2,
    'Eve': 3,
    'Night': 4,
    'Late Night': 5
}

df['order_daynight'] = df['order_daypart'].map(daypart_map)
df['ship_daynight'] = df['ship_daypart'].map(daypart_map)


In [10]:
df

,payment_type,profit_per_order,sales_per_customer,category_name,customer_segment,department_name,latitude,longitude,market,order_city,...,order_to_shipment_days,order_dayofweek,shipping_dayofweek,order_hour,shipping_hour,order_shipping_time,order_daypart,ship_daypart,order_daynight,ship_daynight
0,PAYMENT,101.010895,195.02570,Indoor/Outdoor Games,Home Office,Fan Shop,41.478510,-87.972565,Europe,Viena,...,5,2,0,17,17,120.0,Eve,Eve,3,3
1,TRANSFER,85.423610,245.20793,Indoor/Outdoor Games,Consumer,Fan Shop,18.281605,-66.370510,LATAM,Buenos Aires,...,3,1,4,23,23,72.0,Night,Night,4,4
2,PAYMENT,261.173770,456.55527,Cardio Equipment,Consumer,Footwear,18.281320,-71.919000,Europe,Bruges,...,3,2,5,13,13,72.0,Noon,Noon,2,2
3,DEBIT,-52.374670,191.65901,Water Sports,Consumer,Fan Shop,18.289013,-66.370520,LATAM,Rancagua,...,2,6,1,3,3,48.0,Late Night,Late Night,5,5
4,DEBIT,55.085342,187.45561,Water Sports,Consumer,Fan Shop,18.227660,-66.370530,USCA,New York City,...,2,3,5,14,14,48.0,Noon,Noon,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15543,CASH,72.536430,269.98000,Cleats,Consumer,Apparel,41.894955,-81.512120,LATAM,Panama City,...,4,2,6,2,2,96.0,Late Night,Late Night,5,5
15544,PAYMENT,160.320020,395.98000,Fishing,Corporate,Fan Shop,30.351015,-104.869850,USCA,Highland Park,...,2,2,4,17,17,48.0,Eve,Eve,3,3
15545,DEBIT,9.606151,21.29201,Golf Balls,Consumer,Outdoors,18.223124,-66.370575,LATAM,Chapeco,...,5,0,5,9,9,120.0,Morning,Morning,1,1
15546,TRANSFER,-75.001240,246.49313,Indoor/Outdoor Games,Home Office,Fan Shop,34.127888,-117.257670,Pacific Asia,Amman,...,5,1,6,0,0,120.0,Late Night,Late Night,5,5


In [11]:
df.columns

Index(['payment_type', 'profit_per_order', 'sales_per_customer',
       'category_name', 'customer_segment', 'department_name', 'latitude',
       'longitude', 'market', 'order_city', 'order_country', 'order_date',
       'order_item_discount', 'order_item_discount_rate',
       'order_item_product_price', 'order_item_profit_ratio',
       'order_item_quantity', 'sales', 'order_item_total_amount',
       'order_profit_per_order', 'order_region', 'order_state', 'order_status',
       'product_name', 'shipping_date', 'shipping_mode', 'label',
       'dest_latitude', 'dest_longitude', 'store_to_order_distance_km',
       'distance_normalized', 'customer_country', 'is_international',
       'customer_city', 'customer_state', 'order_to_shipment_days',
       'order_dayofweek', 'shipping_dayofweek', 'order_hour', 'shipping_hour',
       'order_shipping_time', 'order_daypart', 'ship_daypart',
       'order_daynight', 'ship_daynight'],
      dtype='object')

In [12]:
shipping_mode_to_days = {
    'Same Day': 0,
    'First Class': 1,   # Midpoint of 1–5 days
    'Second Class': 2,  # Midpoint of ~2–5 days
    'Standard Class': 4  # Midpoint of 3–7 days
}

# Step 2: Map it
df['order_to_shipment_planned_days'] = df['shipping_mode'].map(shipping_mode_to_days)

In [13]:
df['shipment_delay_days'] = df['order_to_shipment_days'] - df['order_to_shipment_planned_days']

In [14]:
df

,payment_type,profit_per_order,sales_per_customer,category_name,customer_segment,department_name,latitude,longitude,market,order_city,...,shipping_dayofweek,order_hour,shipping_hour,order_shipping_time,order_daypart,ship_daypart,order_daynight,ship_daynight,order_to_shipment_planned_days,shipment_delay_days
0,PAYMENT,101.010895,195.02570,Indoor/Outdoor Games,Home Office,Fan Shop,41.478510,-87.972565,Europe,Viena,...,0,17,17,120.0,Eve,Eve,3,3,4,1
1,TRANSFER,85.423610,245.20793,Indoor/Outdoor Games,Consumer,Fan Shop,18.281605,-66.370510,LATAM,Buenos Aires,...,4,23,23,72.0,Night,Night,4,4,4,-1
2,PAYMENT,261.173770,456.55527,Cardio Equipment,Consumer,Footwear,18.281320,-71.919000,Europe,Bruges,...,5,13,13,72.0,Noon,Noon,2,2,4,-1
3,DEBIT,-52.374670,191.65901,Water Sports,Consumer,Fan Shop,18.289013,-66.370520,LATAM,Rancagua,...,1,3,3,48.0,Late Night,Late Night,5,5,4,-2
4,DEBIT,55.085342,187.45561,Water Sports,Consumer,Fan Shop,18.227660,-66.370530,USCA,New York City,...,5,14,14,48.0,Noon,Noon,2,2,4,-2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15543,CASH,72.536430,269.98000,Cleats,Consumer,Apparel,41.894955,-81.512120,LATAM,Panama City,...,6,2,2,96.0,Late Night,Late Night,5,5,2,2
15544,PAYMENT,160.320020,395.98000,Fishing,Corporate,Fan Shop,30.351015,-104.869850,USCA,Highland Park,...,4,17,17,48.0,Eve,Eve,3,3,4,-2
15545,DEBIT,9.606151,21.29201,Golf Balls,Consumer,Outdoors,18.223124,-66.370575,LATAM,Chapeco,...,5,9,9,120.0,Morning,Morning,1,1,2,3
15546,TRANSFER,-75.001240,246.49313,Indoor/Outdoor Games,Home Office,Fan Shop,34.127888,-117.257670,Pacific Asia,Amman,...,6,0,0,120.0,Late Night,Late Night,5,5,4,1


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15548 entries, 0 to 15547
Data columns (total 47 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   payment_type                    15548 non-null  object        
 1   profit_per_order                15548 non-null  float64       
 2   sales_per_customer              15548 non-null  float64       
 3   category_name                   15548 non-null  object        
 4   customer_segment                15548 non-null  object        
 5   department_name                 15548 non-null  object        
 6   latitude                        15548 non-null  float64       
 7   longitude                       15548 non-null  float64       
 8   market                          15548 non-null  object        
 9   order_city                      15548 non-null  object        
 10  order_country                   15548 non-null  object        
 11  or

In [16]:
#we need to remove some features because we derived some features from them
cols_to_drop = [
    "latitude", "longitude",                         # raw location data → already captured via distance_normalized
    "dest_latitude", "dest_longitude",               # destination coords → redundant after distance calc
    "store_to_order_distance_km",                    # normalized version used → raw not needed
    "shipping_date", "order_date",                   # derived hour week day night part and others so we dont need anymore
    "order_daypart", "ship_daypart",                 # removing text based daypart because we have in numerical form
    "order_status"
    # Removed 'order_status' feature because:
# - It is a high-level business process label (e.g., COMPLETE, CANCELED, PENDING)
# - It may **leak information about delivery outcome** (e.g., CANCELED may imply no delivery)
# - It is **not a causal or predictive feature** for forecasting delays in real time
# - It can introduce **data leakage** if populated after shipment events
# - Retaining it may lead to **overfitting** and overly optimistic performance
]

# Drop those columns
df = df.drop(columns=cols_to_drop)

# Check remaining columns
print("Remaining columns:", df.columns.tolist())

Remaining columns: ['payment_type', 'profit_per_order', 'sales_per_customer', 'category_name', 'customer_segment', 'department_name', 'market', 'order_city', 'order_country', 'order_item_discount', 'order_item_discount_rate', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total_amount', 'order_profit_per_order', 'order_region', 'order_state', 'product_name', 'shipping_mode', 'label', 'distance_normalized', 'customer_country', 'is_international', 'customer_city', 'customer_state', 'order_to_shipment_days', 'order_dayofweek', 'shipping_dayofweek', 'order_hour', 'shipping_hour', 'order_shipping_time', 'order_daynight', 'ship_daynight', 'order_to_shipment_planned_days', 'shipment_delay_days']


In [17]:
df.shape

(15548, 37)

STATISTICAL TESTING

In [18]:
import pandas as pd
import numpy as np
from scipy.stats import f_oneway, kruskal, chi2_contingency
import warnings

warnings.filterwarnings("ignore")  # Hide warnings for cleaner output

TARGET_COL = 'label'

# Step 3: Split into numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

In [19]:
df.info()
df['label'] = df['label'].astype('category')
print(df['label'].dtype)  # should show 'category'

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15548 entries, 0 to 15547
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   payment_type                    15548 non-null  object 
 1   profit_per_order                15548 non-null  float64
 2   sales_per_customer              15548 non-null  float64
 3   category_name                   15548 non-null  object 
 4   customer_segment                15548 non-null  object 
 5   department_name                 15548 non-null  object 
 6   market                          15548 non-null  object 
 7   order_city                      15548 non-null  object 
 8   order_country                   15548 non-null  object 
 9   order_item_discount             15548 non-null  float64
 10  order_item_discount_rate        15548 non-null  float64
 11  order_item_product_price        15548 non-null  float64
 12  order_item_profit_ratio         

In [20]:
from scipy.stats import f_oneway, kruskal, chi2_contingency
import pandas as pd

# Make sure target is not in the list
if TARGET_COL in num_cols: num_cols.remove(TARGET_COL)
if TARGET_COL in cat_cols: cat_cols.remove(TARGET_COL)

# Step 4: Run statistical tests

# --- Numerical features ---
print("\n🔍 NUMERICAL FEATURES vs MULTICLASS TARGET (ANOVA + Kruskal-Wallis)")
print("------------------------------------------------------")

anova_kruskal_results = []

for col in num_cols:
    try:
        groups = [df[df[TARGET_COL] == label][col].dropna() for label in sorted(df[TARGET_COL].unique())]

        # ANOVA
        f_stat, p_anova = f_oneway(*groups)

        # Kruskal-Wallis
        h_stat, p_kruskal = kruskal(*groups)

        anova_kruskal_results.append((col, p_anova, p_kruskal))

    except Exception as e:
        print(f"{col:<30} | Error in test: {e}")

# Sort by ANOVA p-value
anova_kruskal_results.sort(key=lambda x: x[1])

# Print sorted results
for col, p_anova, p_kruskal in anova_kruskal_results:
    print(f"{col:<30} | ANOVA p = {p_anova:.4f} | Kruskal p = {p_kruskal:.4f}")


# --- Categorical features ---
print("\n🔍 CATEGORICAL FEATURES vs MULTICLASS TARGET (Chi-Square)")
print("------------------------------------------------------")

chi2_results = []

for col in cat_cols:
    try:
        contingency = pd.crosstab(df[col], df[TARGET_COL])
        chi2, p_chi, _, _ = chi2_contingency(contingency)
        chi2_results.append((col, p_chi))
    except Exception as e:
        print(f"{col:<30} | Error in test: {e}")

# Sort by Chi-square p-value
chi2_results.sort(key=lambda x: x[1])

# Print sorted results
for col, p_chi in chi2_results:
    print(f"{col:<30} | Chi-square p = {p_chi:.4f}")


🔍 NUMERICAL FEATURES vs MULTICLASS TARGET (ANOVA + Kruskal-Wallis)
------------------------------------------------------
order_to_shipment_planned_days | ANOVA p = 0.0000 | Kruskal p = 0.0000
shipment_delay_days            | ANOVA p = 0.0000 | Kruskal p = 0.0000
order_shipping_time            | ANOVA p = 0.0004 | Kruskal p = 0.0002
order_to_shipment_days         | ANOVA p = 0.0004 | Kruskal p = 0.0002
order_item_discount_rate       | ANOVA p = 0.0046 | Kruskal p = 0.0055
order_item_discount            | ANOVA p = 0.0102 | Kruskal p = 0.0032
order_daynight                 | ANOVA p = 0.0822 | Kruskal p = 0.0821
sales                          | ANOVA p = 0.1432 | Kruskal p = 0.4788
order_item_quantity            | ANOVA p = 0.1545 | Kruskal p = 0.1904
ship_daynight                  | ANOVA p = 0.1743 | Kruskal p = 0.1741
order_item_product_price       | ANOVA p = 0.2023 | Kruskal p = 0.1123
order_item_total_amount        | ANOVA p = 0.2596 | Kruskal p = 0.6953
sales_per_customer       

In [21]:
# List of features to drop based on statistical insignificance
features_to_drop = [
    # Numerical (p > 0.05 in both ANOVA & Kruskal)
    # 'sales',                      # High p-values ⇒ no strong relation with target
    # 'order_item_quantity',        # Insignificant across groups
    # 'ship_daynight',              # Doesn't differentiate target classes
    # 'order_item_product_price',   # Weak signal
    'order_item_total_amount',    # No useful variation
    'sales_per_customer',         # Not distinguishing
    # 'distance_normalized',        # No evidence of predictive signal
    # 'order_item_profit_ratio',    # Doesn't help separate classes
    # 'profit_per_order',           # Weak effect
    # 'order_profit_per_order',     # Not informative

    # Categorical (Chi-Square p > 0.05)
    # 'customer_country',           # Weak association with target
    # 'customer_city',              # Too granular or noisy
    # 'market',                     # No significant influence
    # 'product_name',               # Too detailed; likely too many categories
    # 'order_city',                 # Weak effect
    # 'payment_type',               # Not linked to target variation
    # 'category_name',              # No major effect
    # 'order_country',              # Weak correlation
    # 'customer_segment',           # Not separating classes well
    # 'order_state',                # Statistically irrelevant
    # 'order_region',              # Weak link to target
    'department_name'            # No impact seen
]

# Drop them from the dataframe
df = df.drop(columns=features_to_drop)

In [22]:
df.shape

(15548, 34)

CORRELATIONS

In [23]:
df['label'] = df['label'].astype('int64')

In [24]:
numerical_data = df.select_dtypes(include = ['number']).copy()

In [25]:
corr_pearson = numerical_data.corr(method = 'pearson')
corr_pearson

,profit_per_order,order_item_discount,order_item_discount_rate,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_profit_per_order,label,distance_normalized,order_to_shipment_days,order_dayofweek,shipping_dayofweek,order_hour,shipping_hour,order_shipping_time,order_daynight,ship_daynight,order_to_shipment_planned_days,shipment_delay_days
profit_per_order,1.000000,0.067059,-0.004241,0.076971,0.716811,0.017708,0.104170,0.862400,-0.003622,-0.001632,0.011059,-0.002059,0.008444,-0.003243,0.001379,0.011351,-0.007222,-0.005494,-0.003189,0.010794
order_item_discount,0.067059,1.000000,0.666819,0.486559,0.000952,0.068011,0.616807,0.078983,0.023625,-0.005663,-0.017514,0.014541,-0.010775,0.013795,0.014313,-0.018166,-0.008452,-0.006236,-0.003689,-0.011397
order_item_discount_rate,-0.004241,0.666819,1.000000,0.009111,-0.007255,-0.005060,0.003252,-0.015064,0.026067,-0.004736,-0.004173,-0.002984,-0.015029,0.006195,0.003691,-0.004638,-0.007529,-0.008080,-0.013362,0.005421
order_item_product_price,0.076971,0.486559,0.009111,1.000000,-0.008373,-0.483853,0.769677,0.100407,0.014336,0.014070,-0.035473,0.011343,-0.008820,-0.000480,0.007710,-0.034617,-0.008099,-0.002732,-0.002369,-0.026410
order_item_profit_ratio,0.716811,0.000952,-0.007255,-0.008373,1.000000,0.009819,-0.001352,0.839579,0.006861,0.005717,-0.001113,-0.002295,0.014263,-0.000580,0.003323,-0.001057,-0.000103,-0.001584,-0.001518,0.000112
order_item_quantity,0.017708,0.068011,-0.005060,-0.483853,0.009819,1.000000,0.128958,0.029091,-0.003375,-0.020397,0.022588,0.005124,0.017290,0.004514,0.000112,0.020822,0.001791,-0.001401,0.001183,0.017029
sales,0.104170,0.616807,0.003252,0.769677,-0.001352,0.128958,1.000000,0.140681,0.013606,-0.003087,-0.023908,0.015465,0.002209,0.004075,0.011370,-0.024399,-0.008112,-0.002821,-0.001648,-0.017767
order_profit_per_order,0.862400,0.078983,-0.015064,0.100407,0.839579,0.029091,0.140681,1.000000,-0.000378,0.006093,-0.002659,-0.002380,0.015167,-0.000134,0.002967,-0.002327,-0.002420,-0.003362,-0.006764,0.002313
label,-0.003622,0.023625,0.026067,0.014336,0.006861,-0.003375,0.013606,-0.000378,1.000000,0.007128,-0.019168,0.004988,0.014300,0.002394,0.000883,-0.018488,0.008469,0.007124,-0.452746,0.279981
distance_normalized,-0.001632,-0.005663,-0.004736,0.014070,0.005717,-0.020397,-0.003087,0.006093,0.007128,1.000000,0.002564,-0.012056,0.000412,-0.000690,0.000816,0.003155,0.014658,0.013331,-0.013028,0.010511


In [26]:
corr_pearson["label"].sort_values(ascending = False)

label                             1.000000
shipment_delay_days               0.279981
order_item_discount_rate          0.026067
order_item_discount               0.023625
order_item_product_price          0.014336
shipping_dayofweek                0.014300
sales                             0.013606
order_daynight                    0.008469
distance_normalized               0.007128
ship_daynight                     0.007124
order_item_profit_ratio           0.006861
order_dayofweek                   0.004988
order_hour                        0.002394
shipping_hour                     0.000883
order_profit_per_order           -0.000378
order_item_quantity              -0.003375
profit_per_order                 -0.003622
order_shipping_time              -0.018488
order_to_shipment_days           -0.019168
order_to_shipment_planned_days   -0.452746
Name: label, dtype: float64

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15548 entries, 0 to 15547
Data columns (total 34 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   payment_type                    15548 non-null  object 
 1   profit_per_order                15548 non-null  float64
 2   category_name                   15548 non-null  object 
 3   customer_segment                15548 non-null  object 
 4   market                          15548 non-null  object 
 5   order_city                      15548 non-null  object 
 6   order_country                   15548 non-null  object 
 7   order_item_discount             15548 non-null  float64
 8   order_item_discount_rate        15548 non-null  float64
 9   order_item_product_price        15548 non-null  float64
 10  order_item_profit_ratio         15548 non-null  float64
 11  order_item_quantity             15548 non-null  float64
 12  sales                           

OUTLIERS

In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis, shapiro, normaltest, probplot

In [29]:
cols_to_analyze = [
    'order_item_discount',
    'order_item_discount_rate',
    'order_item_product_price',
    'order_item_profit_ratio',
    'order_item_quantity',
    'profit_per_order',
    'order_profit_per_order',
    'sales',
    'distance_normalized',
    'order_to_shipment_days',
    'order_to_shipment_planned_days',
    'shipment_delay_days',
    'order_shipping_time'
]

for col in cols_to_analyze:
    data = df[col].dropna()
    
    print(f"\n Feature: {col}")
    print("--------------------------------------------------")
    
    # Numerical Metrics
    sk = skew(data)
    kt = kurtosis(data)
    stat_shapiro, p_shapiro = shapiro(data)
    stat_dagostino, p_dagostino = normaltest(data)

    print(f"Skewness       : {sk:.4f} {'(Right Skewed)' if sk > 0.5 else '(Left Skewed)' if sk < -0.5 else '(Approximately Symmetric)'}")
    print(f"Kurtosis       : {kt:.4f} {'(Heavy tails)' if kt > 3 else '(Light tails)' if kt < 3 else '(Normal-like tails)'}")
    print(f"Shapiro-Wilk   : p = {p_shapiro:.4f} {'→ Not Normal' if p_shapiro <= 0.05 else '→ Possibly Normal'}")
    print(f"D’Agostino Test: p = {p_dagostino:.4f} {'→ Not Normal' if p_dagostino <= 0.05 else '→ Possibly Normal'}")

    # # Visualization
    # fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    # sns.histplot(data, kde=True, ax=axes[0], color='steelblue')
    # axes[0].set_title("Histogram + KDE")

    # sns.boxplot(x=data, ax=axes[1], color='lightcoral')
    # axes[1].set_title("Boxplot")

    # probplot(data, dist="norm", plot=axes[2])
    # axes[2].set_title("Q-Q Plot")

    # plt.tight_layout()
    # plt.show()


 Feature: order_item_discount
--------------------------------------------------
Skewness       : 3.0138 (Right Skewed)
Kurtosis       : 26.2731 (Heavy tails)
Shapiro-Wilk   : p = 0.0000 → Not Normal
D’Agostino Test: p = 0.0000 → Not Normal

 Feature: order_item_discount_rate
--------------------------------------------------
Skewness       : 0.3322 (Approximately Symmetric)
Kurtosis       : -0.9336 (Light tails)
Shapiro-Wilk   : p = 0.0000 → Not Normal
D’Agostino Test: p = 0.0000 → Not Normal

 Feature: order_item_product_price
--------------------------------------------------
Skewness       : 2.7514 (Right Skewed)
Kurtosis       : 18.2868 (Heavy tails)
Shapiro-Wilk   : p = 0.0000 → Not Normal
D’Agostino Test: p = 0.0000 → Not Normal

 Feature: order_item_profit_ratio
--------------------------------------------------
Skewness       : -2.9324 (Left Skewed)
Kurtosis       : 10.6192 (Heavy tails)
Shapiro-Wilk   : p = 0.0000 → Not Normal
D’Agostino Test: p = 0.0000 → Not Normal

 Featu

In [30]:
def count_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return ((df[col] < lower) | (df[col] > upper)).sum()
def count_outliers_zscore(df, col, threshold=3):
    z_scores = (df[col] - df[col].mean()) / df[col].std()
    return (np.abs(z_scores) > threshold).sum()
iqr_cols = [
    'order_item_discount', 'order_item_product_price', 
    'order_item_profit_ratio', 'order_item_quantity', 
    'order_to_shipment_planned_days', 'profit_per_order', 'order_profit_per_order'
]

zscore_cols = [
    'order_item_discount_rate', 'distance_normalized', 
    'order_to_shipment_days', 'shipment_delay_days', 'order_shipping_time'
]

quantile_cols = ['sales']
outlier_counts = {}

for col in iqr_cols:
    count = count_outliers_iqr(df, col)
    outlier_counts[col] = count

for col in zscore_cols:
    count = count_outliers_zscore(df, col)
    outlier_counts[col] = count

# Optional: quantile method (e.g., outside 1st–99th percentile)
def count_outliers_quantile(df, col, lower_q=0.01, upper_q=0.99):
    lower = df[col].quantile(lower_q)
    upper = df[col].quantile(upper_q)
    return ((df[col] < lower) | (df[col] > upper)).sum()

for col in quantile_cols:
    count = count_outliers_quantile(df, col)
    outlier_counts[col] = count

# Show all
import pandas as pd
outlier_df = pd.DataFrame(list(outlier_counts.items()), columns=["Feature", "Outlier Count"])
outlier_df["Outlier %"] = (outlier_df["Outlier Count"] / len(df) * 100).round(2)
print(outlier_df)

                           Feature  Outlier Count  Outlier %
0              order_item_discount            614       3.95
1         order_item_product_price            134       0.86
2          order_item_profit_ratio           1520       9.78
3              order_item_quantity              0       0.00
4   order_to_shipment_planned_days              0       0.00
5                 profit_per_order           1640      10.55
6           order_profit_per_order           1620      10.42
7         order_item_discount_rate              0       0.00
8              distance_normalized              0       0.00
9           order_to_shipment_days              0       0.00
10             shipment_delay_days              0       0.00
11             order_shipping_time              0       0.00
12                           sales            179       1.15


In [31]:
def cap_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])
    return df

def cap_outliers_zscore(df, col, threshold=3):
    mean = df[col].mean()
    std = df[col].std()
    lower = mean - threshold * std
    upper = mean + threshold * std
    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])
    return df

def cap_outliers_quantile(df, col, lower_quantile=0.01, upper_quantile=0.99):
    lower = df[col].quantile(lower_quantile)
    upper = df[col].quantile(upper_quantile)
    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])
    return df

| Feature                            | Skewness     | Tails | Method to Use   |
| ---------------------------------- | ------------ | ----- | --------------- |
| order\_item\_discount              | Right Skewed | Heavy | IQR or Quantile |
| order\_item\_discount\_rate        | Symmetric    | Light | Z-score         |
| order\_item\_product\_price        | Right Skewed | Heavy | IQR or Quantile |
| order\_item\_profit\_ratio         | Left Skewed  | Heavy | IQR or Quantile |
| order\_item\_quantity              | Right Skewed | Light | IQR             |
| sales                              | Right Skewed | Heavy | Quantile        |
| distance\_normalized               | Symmetric    | Light | Z-score         |
| order\_to\_shipment\_days          | Symmetric    | Light | Z-score         |
| order\_to\_shipment\_planned\_days | Left Skewed  | Light | IQR or Quantile |
| shipment\_delay\_days              | Symmetric    | Light | Z-score         |


In [32]:
# Capping strategies mapped
iqr_cols = [
    'order_item_discount', 'order_item_product_price', 'order_item_profit_ratio',
    'order_item_quantity', 'order_to_shipment_planned_days'
]
zscore_cols = [
    'order_item_discount_rate', 'distance_normalized',
    'order_to_shipment_days', 'shipment_delay_days', 'order_shipping_time'
]
quantile_cols = ['sales', 'profit_per_order', 'order_profit_per_order']  # heavy skew + large values

# Apply capping
for col in iqr_cols:
    df = cap_outliers_iqr(df, col)

for col in zscore_cols:
    df = cap_outliers_zscore(df, col)

for col in quantile_cols:
    df = cap_outliers_quantile(df, col)

In [33]:
df.columns

Index(['payment_type', 'profit_per_order', 'category_name', 'customer_segment',
       'market', 'order_city', 'order_country', 'order_item_discount',
       'order_item_discount_rate', 'order_item_product_price',
       'order_item_profit_ratio', 'order_item_quantity', 'sales',
       'order_profit_per_order', 'order_region', 'order_state', 'product_name',
       'shipping_mode', 'label', 'distance_normalized', 'customer_country',
       'is_international', 'customer_city', 'customer_state',
       'order_to_shipment_days', 'order_dayofweek', 'shipping_dayofweek',
       'order_hour', 'shipping_hour', 'order_shipping_time', 'order_daynight',
       'ship_daynight', 'order_to_shipment_planned_days',
       'shipment_delay_days'],
      dtype='object')

In [34]:
df.shape

(15548, 34)

In [35]:
df.to_csv("new_engineered_features.csv", index=False)

FEATURE SELECTION

In [36]:
numeric_mask1 = df.select_dtypes(include='number').columns
numeric_features = df[numeric_mask1]

In [37]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

var_th = VarianceThreshold(threshold=0.05)
var_th.fit(numeric_features)

# See which columns are removed
variances = var_th.variances_
cols = numeric_features.columns

# Create a summary DataFrame
var_df = pd.DataFrame({
    'Feature': cols,
    'Variance': variances,
    'Keep': variances >= 0.05
})

print(var_df.sort_values(by="Variance"))


                           Feature      Variance   Keep
2         order_item_discount_rate      0.005058  False
9              distance_normalized      0.053647   True
4          order_item_profit_ratio      0.063090   True
8                            label      0.700232   True
18  order_to_shipment_planned_days      1.876704   True
5              order_item_quantity      2.106336   True
10          order_to_shipment_days      2.743522   True
17                   ship_daynight      2.931387   True
16                  order_daynight      2.936905   True
11                 order_dayofweek      4.020302   True
12              shipping_dayofweek      4.046742   True
19             shipment_delay_days      4.417809   True
13                      order_hour     47.957037   True
14                   shipping_hour     48.009376   True
1              order_item_discount    331.184739   True
15             order_shipping_time   1486.338423   True
0                 profit_per_order   6719.417586

In [38]:
X_num = numeric_features.drop(columns = ["label"]) 
y_num = numeric_features["label"]  

In [39]:
import scipy
import numpy as np

print(f"{'col':<30} {'corr':<10} {'pval':<10}")
for col in X_num.columns:
    corr = np.nan
    pval = np.nan
    if X_num[col].nunique() > 1:  # Check if column has more than one unique value
        corr, pval = scipy.stats.pearsonr(X_num[col], y_num)
    
    print(f"{col:<30} {corr:>10.2f} {pval:>10.4f}")

col                            corr       pval      
profit_per_order                     0.00     0.8248
order_item_discount                  0.02     0.0030
order_item_discount_rate             0.03     0.0012
order_item_product_price             0.01     0.2020
order_item_profit_ratio              0.01     0.2917
order_item_quantity                 -0.00     0.6739
sales                                0.01     0.2330
order_profit_per_order               0.00     0.7410
distance_normalized                  0.01     0.3742
order_to_shipment_days              -0.02     0.0168
order_dayofweek                      0.00     0.5340
shipping_dayofweek                   0.01     0.0746
order_hour                           0.00     0.7654
shipping_hour                        0.00     0.9124
order_shipping_time                 -0.02     0.0212
order_daynight                       0.01     0.2910
ship_daynight                        0.01     0.3744
order_to_shipment_planned_days      -0.45     

In [40]:
from sklearn.feature_selection import mutual_info_regression
list(zip(X_num.columns, mutual_info_regression(X_num, y_num, random_state=123, n_neighbors=3)))

[('profit_per_order', np.float64(0.0017526346231822032)),
 ('order_item_discount', np.float64(0.004995407057550416)),
 ('order_item_discount_rate', np.float64(0.003167540518274947)),
 ('order_item_product_price', np.float64(0.015344805525519867)),
 ('order_item_profit_ratio', np.float64(0.0)),
 ('order_item_quantity', np.float64(0.0)),
 ('sales', np.float64(0.0)),
 ('order_profit_per_order', np.float64(0.005387195853295168)),
 ('distance_normalized', np.float64(0.007326810415596796)),
 ('order_to_shipment_days', np.float64(0.0)),
 ('order_dayofweek', np.float64(0.0)),
 ('shipping_dayofweek', np.float64(0.0025768953721270194)),
 ('order_hour', np.float64(0.005548189143507187)),
 ('shipping_hour', np.float64(0.0)),
 ('order_shipping_time', np.float64(0.004134502066158063)),
 ('order_daynight', np.float64(0.012548492009553769)),
 ('ship_daynight', np.float64(0.0)),
 ('order_to_shipment_planned_days', np.float64(0.20302117576404566)),
 ('shipment_delay_days', np.float64(0.07626887409075422

In [41]:
df.dtypes

payment_type                       object
profit_per_order                  float64
category_name                      object
customer_segment                   object
market                             object
order_city                         object
order_country                      object
order_item_discount               float64
order_item_discount_rate          float64
order_item_product_price          float64
order_item_profit_ratio           float64
order_item_quantity               float64
sales                             float64
order_profit_per_order            float64
order_region                       object
order_state                        object
product_name                       object
shipping_mode                      object
label                               int64
distance_normalized               float64
customer_country                   object
is_international                     bool
customer_city                      object
customer_state                    

In [42]:
df.shape

(15548, 34)

In [43]:
df["is_international"].value_counts()

is_international
True    15548
Name: count, dtype: int64

In [44]:
features_to_remove = ['order_item_discount_rate', 'is_international']
df = df.drop(columns=features_to_remove)

In [45]:
df.shape

(15548, 32)

In [46]:
df.to_csv("new_engineered_features2.csv", index = False)